# SALES_ML Feature Store セットアップ

FOODEX_DEMO.SALES_ML スキーマに Feature Store を0から構築します。

### 作成するオブジェクト
| オブジェクト | 名前 | 説明 |
|------------|------|------|
| Schema | SALES_ML | ML関連オブジェクト集約スキーマ |
| Entity | CATEGORY_DATE_ENTITY | カテゴリ×日付の複合エンティティ |
| Feature View | SALES_FORECAST_FV (v2) | LAG/移動平均/カレンダー特徴量 |

### ソースデータ（BUYER_AGENTスキーマ）
- `FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS`
- `FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER`

## Step 1: セッション初期化 & スキーマ作成

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()

session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
session.sql("""
    CREATE SCHEMA IF NOT EXISTS FOODEX_DEMO.SALES_ML
        COMMENT = 'ML関連オブジェクト集約スキーマ'
""").collect()
session.sql("USE SCHEMA SALES_ML").collect()

print("Database: FOODEX_DEMO")
print("Schema:   SALES_ML")
print("Warehouse: COMPUTE_WH")

## Step 2: Feature Store 初期化 & Entity 登録

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="FOODEX_DEMO",
    name="SALES_ML",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature Store initialized: FOODEX_DEMO.SALES_ML")

In [ ]:
category_date_entity = Entity(
    name="CATEGORY_DATE_ENTITY",
    join_keys=["CATEGORY_MEDIUM", "SALES_DATE"],
    desc="Category and date composite entity for sales forecasting"
)

try:
    fs.register_entity(category_date_entity)
    print("Entity 'CATEGORY_DATE_ENTITY' registered successfully.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Entity 'CATEGORY_DATE_ENTITY' already exists, continuing...")
    else:
        raise e

print(f"  join_keys: {category_date_entity.join_keys}")

## Step 3: 特徴量クエリ定義

BUYER_AGENTスキーマのPOSデータから以下の特徴量を生成します:

| 特徴量 | 説明 |
|--------|------|
| DAILY_SALES | 日次売上額 |
| TXN_COUNT | トランザクション数 |
| TOTAL_QTY | 合計数量 |
| LAG_1〜LAG_14 | 1/2/3/7/14日前の売上 |
| MA_7 | 7日移動平均 |
| DAY_OF_WEEK | 曜日（0-6） |
| IS_WEEKEND | 週末フラグ |
| MONTH_NUM | 月 |
| QUARTER_NUM | 四半期 |

In [ ]:
feature_query = """
WITH daily_agg AS (
    SELECT 
        t.TRANSACTION_DATE AS SALES_DATE,
        p.CATEGORY_MEDIUM,
        SUM(t.SALES_AMOUNT) AS DAILY_SALES,
        COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
        SUM(t.QUANTITY) AS TOTAL_QTY
    FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
    JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
        ON t.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.CATEGORY_MEDIUM IS NOT NULL
    GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
),
lag_features AS (
    SELECT 
        SALES_DATE,
        CATEGORY_MEDIUM,
        DAILY_SALES,
        TXN_COUNT,
        TOTAL_QTY,
        LAG(DAILY_SALES, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_1,
        LAG(DAILY_SALES, 2) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_2,
        LAG(DAILY_SALES, 3) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_3,
        LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_7,
        LAG(DAILY_SALES, 14) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_14,
        AVG(DAILY_SALES) OVER (
            PARTITION BY CATEGORY_MEDIUM 
            ORDER BY SALES_DATE 
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS MA_7,
        DAYOFWEEK(SALES_DATE) AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(SALES_DATE) IN (0, 6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        MONTH(SALES_DATE) AS MONTH_NUM,
        QUARTER(SALES_DATE) AS QUARTER_NUM
    FROM daily_agg
)
SELECT * FROM lag_features
WHERE LAG_14 IS NOT NULL
"""

feature_df = session.sql(feature_query)

print(f"レコード数: {feature_df.count()}")
print(f"カラム: {feature_df.columns}")
feature_df.show(5)

## Step 4: Feature View 登録 (SALES_FORECAST_FV v2)

In [ ]:
sales_fv = FeatureView(
    name="SALES_FORECAST_FV",
    entities=[category_date_entity],
    feature_df=feature_df,
    refresh_freq="1 day",
    desc="Sales forecast features with lag, moving average, and calendar features"
)

try:
    sales_fv = fs.register_feature_view(
        feature_view=sales_fv,
        version="v2",
        block=True
    )
    print("Feature View registered successfully!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Feature View already exists, retrieving...")
        sales_fv = fs.get_feature_view("SALES_FORECAST_FV", "v2")
    else:
        raise e

print(f"\n=== SALES_FORECAST_FV v2 ===")
print(f"  Schema:       FOODEX_DEMO.SALES_ML")
print(f"  Entity:       CATEGORY_DATE_ENTITY")
print(f"  Join Keys:    [CATEGORY_MEDIUM, SALES_DATE]")
print(f"  Refresh:      1 day (毎日自動更新)")

## Step 5: 検証 - Feature View からデータ読み取り

In [ ]:
print("=== 登録済み Feature View 一覧 ===")
fv_list = fs.list_feature_views().to_pandas()
for fv in fv_list.itertuples():
    print(f"  - {fv.NAME} (version: {fv.VERSION})")

print("\n=== Feature View データ読み取りテスト ===")
test_df = fs.read_feature_view(sales_fv)
print(f"  レコード数: {test_df.count()}")
print(f"  期間: {test_df.select(F.min('SALES_DATE'), F.max('SALES_DATE')).collect()}")
print(f"  カテゴリ数: {test_df.select(F.count_distinct('CATEGORY_MEDIUM')).collect()[0][0]}")
test_df.show(5)

In [ ]:
print("=" * 50)
print("  Feature Store セットアップ完了")
print("=" * 50)
print(f"\n[作成オブジェクト - FOODEX_DEMO.SALES_ML]")
print(f"  Entity:       CATEGORY_DATE_ENTITY")
print(f"  Feature View: SALES_FORECAST_FV (v2)")
print(f"  Refresh:      毎日自動更新")
print(f"\n[特徴量一覧]")
print(f"  売上: DAILY_SALES, TXN_COUNT, TOTAL_QTY")
print(f"  ラグ: LAG_1, LAG_2, LAG_3, LAG_7, LAG_14")
print(f"  移動平均: MA_7")
print(f"  カレンダー: DAY_OF_WEEK, IS_WEEKEND, MONTH_NUM, QUARTER_NUM")